# Map ClinVar Info and Protein Feature to Variant

In [1]:
import polars as pl

## 1. Loading data

### 1.1 VarChAMP allele collection

Note: here we loaded all the allele collection produced by VarChAMP so far, but not all of them were necessarily profiled by the three assays (DUAL-IPA, Imaging or PPI) in this manuscript.

#### Sequence confirmation results clean-up

In [2]:
varchamp_seq_code = pl.read_csv("../1_inputs/seq_confirmation_results/sequence_confirmation_class_code.tsv", separator="\t")
with pl.Config(fmt_str_lengths=150):
    display(varchamp_seq_code)

code,description
i64,str
1,"""perfectly validated"""
2,"""partially validated, >= 50% coverage"""
3,"""wild type"""
4,"""partial wild type, >= 50% coverage"""
5,"""target and off-target mutation, >= 50% coverage"""
6,"""off-target mutation, >= 50% coverage"""
7,"""truncated, < 50% coverage"""
99,"""no reads"""


In [3]:
def map_chr(num):
    if num == 23:
        return "X"
    elif num == 24:
        return "Y"
    elif num == 12920:
        return "MT"
    else:
        return str(num)


varchamp_seq_df = pl.read_csv("../1_inputs/seq_confirmation_results/VarChampSeqConfirmationResult.tsv", separator="\t")
## clean up data
varchamp_seq_df = varchamp_seq_df.with_columns(
    pl.col("spdi").str.extract(r"NC_0*([0-9]+)", 1).cast(pl.Int32).alias("chr_num"),
    pl.col("spdi").str.split(":").list.get(1).str.strip_chars().cast(pl.Int64).alias("nuc_loc"),
    pl.col("spdi").str.split(":").list.get(2).str.strip_chars().alias("ref_allele"),
    pl.col("spdi").str.split(":").list.get(3).str.strip_chars().alias("alt_allele"),
).with_columns(
    pl.col("chr_num")
    .map_elements(map_chr, return_dtype=pl.String)
    .str.strip_chars()
    .alias("Chrom")
).rename(
    {
        "orf_id_wt": "orf_id",
        "mutation_id_old": "mut_id"
    }
)

# Convert columns to integer (NULL -> None)
columns_to_cast = [
    "entry_sequence_confirmation_class",
    "db_sequence_confirmation_class",
    "n2h_sequence_confirmation_class",
    "n2h_ref_sequence_confirmation_class",
    "dualip_sequence_confirmation_class",
    "dualip_ref_sequence_confirmation_class",
    "mislocalization_sequence_confirmation_class",
    "mislocalization_ref_sequence_confirmation_class"
]

varchamp_seq_df = varchamp_seq_df.with_columns([
    pl.col(col).cast(pl.Int64, strict=False) for col in columns_to_cast
])

# Step 1: Create Y/N for entry clone
varchamp_seq_df = varchamp_seq_df.with_columns([
    # Entry clone logic
    pl.when(pl.col("entry_sequence_confirmation_class").is_in([1, 2]))
    .then(pl.lit("Y"))
    .when(pl.col("entry_sequence_confirmation_class").is_in([3, 4, 5, 6]))
    .then(pl.lit("N"))
    .when(pl.col("entry_sequence_confirmation_class").is_in([7, 99]) | pl.col("entry_sequence_confirmation_class").is_null())
    .then(
        # If entry is 7/99/null, check if ANY expression vector is 1 or 2
        pl.when(
            pl.col("db_sequence_confirmation_class").is_in([1, 2]) |
            pl.col("n2h_sequence_confirmation_class").is_in([1, 2]) |
            pl.col("dualip_sequence_confirmation_class").is_in([1, 2]) |
            pl.col("mislocalization_sequence_confirmation_class").is_in([1, 2])
        )
        .then(pl.lit("Y"))
        .otherwise(pl.lit("N"))
    )
    .otherwise(pl.lit("N"))  # Fallback (shouldn't be needed)
    .alias("entry_seq_final")
])

# Step 2: Create Y/N for expression vectors (they use entry_seq_final for 7/99/null cases)
varchamp_seq_df = varchamp_seq_df.with_columns([
    # DB sequence confirmation (sqY2H)
    pl.when(pl.col("db_sequence_confirmation_class").is_in([1, 2]) & pl.col('db_sequenced') == 1)
    .then(pl.lit("Y"))
    .when(pl.col("db_sequence_confirmation_class").is_in([3, 4, 5, 6]) & pl.col('db_sequenced') == 1)
    .then(pl.lit("N"))
    .when((pl.col("db_sequence_confirmation_class").is_in([7, 99]) | pl.col("db_sequence_confirmation_class").is_null()) & pl.col('db_sequenced') == 1)
    .then(pl.col("entry_seq_final"))  # Use entry clone final decision
    .otherwise(pl.lit("N"))
    .alias("db_seq_final"),
    
    # N2H sequence confirmation
    pl.when(pl.col("n2h_sequence_confirmation_class").is_in([1, 2]) & pl.col('n2h_sequenced') == 1)
    .then(pl.lit("Y"))
    .when(pl.col("n2h_sequence_confirmation_class").is_in([3, 4, 5, 6]) & pl.col('n2h_sequenced') == 1)
    .then(pl.lit("N"))
    .when((pl.col("n2h_sequence_confirmation_class").is_in([7, 99]) | pl.col("n2h_sequence_confirmation_class").is_null()) & pl.col('n2h_sequenced') == 1)
    .then(pl.col("entry_seq_final"))  # Use entry clone final decision
    .otherwise(pl.lit("N"))
    .alias("n2h_seq_final"),
    
    # DUAL-IPA sequence confirmation
    pl.when(pl.col("dualip_sequence_confirmation_class").is_in([1, 2]) & pl.col('dualip_sequenced') == 1)
    .then(pl.lit("Y"))
    .when(pl.col("dualip_sequence_confirmation_class").is_in([3, 4, 5, 6]) & pl.col('dualip_sequenced') == 1)
    .then(pl.lit("N"))
    .when((pl.col("dualip_sequence_confirmation_class").is_in([7, 99]) | pl.col("dualip_sequence_confirmation_class").is_null()) & pl.col('dualip_sequenced') == 1)
    .then(pl.col("entry_seq_final"))  # Use entry clone final decision
    .otherwise(pl.lit("N"))
    .alias("dualip_seq_final"),
    
    # Mislocalization sequence confirmation
    pl.when(pl.col("mislocalization_sequence_confirmation_class").is_in([1, 2]) & pl.col('mislocalization_sequenced') == 1)
    .then(pl.lit("Y"))
    .when(pl.col("mislocalization_sequence_confirmation_class").is_in([3, 4, 5, 6]) & pl.col('mislocalization_sequenced') == 1)
    .then(pl.lit("N"))
    .when((pl.col("mislocalization_sequence_confirmation_class").is_in([7, 99]) | pl.col("mislocalization_sequence_confirmation_class").is_null()) & pl.col('mislocalization_sequenced') == 1)
    .then(pl.col("entry_seq_final"))  # Use entry clone final decision
    .otherwise(pl.lit("N"))
    .alias("mislocalization_seq_final")
])

# # Define masks for allele filtering
# entry_good = pl.col("entry_sequence_confirmation_class").is_in([1, 2])
# entry_discard = pl.col("entry_sequence_confirmation_class").is_in([3, 4, 5, 6])
# entry_null_like = pl.col("entry_sequence_confirmation_class").is_in([7, 99]) | pl.col("entry_sequence_confirmation_class").is_null()

# # At least one expression vector is good
# any_expression_good = (
#     pl.col("db_sequence_confirmation_class").is_in([1, 2]) |
#     pl.col("n2h_sequence_confirmation_class").is_in([1, 2]) |
#     pl.col("dualip_sequence_confirmation_class").is_in([1, 2]) |
#     pl.col("mislocalization_sequence_confirmation_class").is_in([1, 2])
# )

# # Combine logic
# varchamp_seq_df = varchamp_seq_df.filter(
#     entry_good | (entry_null_like & any_expression_good)
# )
# varchamp_seq_df.filter(
#     pl.col('entry_seq_final') == 'Y'
# )

# display(varchamp_seq_df)
varchamp_seq_df.shape

## Output the final results to a data frame
# varchamp_seq_df.write_csv(
#     "../3_outputs/VarChampSeqConfirmationResult_final_call.tsv",
#     separator="\t"
# )

(8710, 46)

### 1.2 Adding CAVA alleles (for imaging profiles)

In [ ]:
## Loading the corrected variant collection files provided by Tong with CAVA allele set
varchamp_w_seq_df = pl.read_csv("../1_inputs/seq_confirmation_results/varchamp_seq_confirm_alleles_clinvar_gnomad_with_seq_081225.tsv", separator="\t")
varchamp_w_seq_df = varchamp_w_seq_df.join(
    varchamp_seq_df.select(
        ["ccsb_mutation_id"]+[col for col in varchamp_seq_df.columns if col not in varchamp_w_seq_df.columns]
    ),
    on="ccsb_mutation_id",
    how="left"
).with_columns(
    pl.col('symbol').str.replace_all(r'[^a-zA-Z0-9-]', '').alias('symbol'),
    pl.col('aa_change').str.replace_all(r'[^a-zA-Z0-9-]', '').alias('aa_change')
).with_columns(
    pl.concat_str([pl.col("symbol"), pl.col("aa_change")], separator="_").alias("gene_variant")
)
varchamp_w_seq_df
# varchamp_w_seq_df.write_csv("../3_outputs/VarChAMP_Allele_Collection_1Perc.tsv", separator="\t")

## 2. Mapping Variant Info

### 2.1 Mapping ClinVar and dbNSFP data

In [6]:
processed_clinvar_parq = "../3_outputs/processed_data/clinvar/clinvar_processed.parquet"
clinvar = pl.scan_parquet(processed_clinvar_parq)
# clinvar_snv = clinvar.filter(pl.col("Type")=='single nucleotide variant') ##.select(pl.col("Type")).unique().collect().to_numpy()
# clinvar_snv.collect().shape

varchamp_w_seq_clinvar = varchamp_w_seq_df.join(
    clinvar.collect(),
    left_on="spdi",
    right_on="SPDI",
    how="left"
).with_columns(
    pl.col("clinvar_clnsig_clean").cast(pl.String).fill_null("Others").alias("clinvar_clnsig_clean"),
    pl.col("clinvar_clnsig_clean_pp_strict").cast(pl.String).fill_null("Others").alias("clinvar_clnsig_clean_pp_strict")
)

In [7]:
## Loading the deduplicated clinvar dbnsfp structure annotation file
df_clinvar_dedup_struct = pl.read_parquet(f"../3_outputs/processed_data/dbnsfp/dbNSFP5.0a_variant.clin_var_re-annot_pdb_dedup_variants.parquet").drop("clinvar_clnsig")
df_clinvar_dedup_struct.head()

clinvar_id,#chr,pos(1-based),ref,alt,aaref,aaalt,rs_dbSNP,hg19_chr,hg19_pos(1-based),hg18_chr,hg18_pos(1-based),aapos,genename,Ensembl_geneid,Ensembl_transcriptid,Ensembl_proteinid,Uniprot_acc,Uniprot_entry,HGVSc_snpEff,HGVSp_snpEff,HGVSc_VEP,HGVSp_VEP,APPRIS,GENCODE_basic,TSL,VEP_canonical,MANE,cds_strand,refcodon,codonpos,codon_degeneracy,Ancestral_allele,AltaiNeandertal,Denisova,VindijiaNeandertal,ChagyrskayaNeandertal,…,LastEvaluated,RS# (dbSNP),nsv/esv (dbVar),RCVaccession,PhenotypeIDS,PhenotypeList,Origin,OriginSimple,Assembly,ChromosomeAccession,Chromosome,Start,Stop,Cytogenetic,ReviewStatus,NumberSubmitters,Guidelines,TestedInGTR,OtherIDs,SubmitterCategories,VariationID,PositionVCF,ReferenceAlleleVCF,AlternateAlleleVCF,SomaticClinicalImpact,SomaticClinicalImpactLastEvaluated,ReviewStatusClinicalImpact,Oncogenicity,OncogenicityLastEvaluated,ReviewStatusOncogenicity,clinvar_nt_change,clinvar_aa_change,SPDI,RefSeq_mRNA,StarStatus,clinvar_clnsig_clean,clinvar_clnsig_clean_pp_strict
str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,…,str,i64,str,str,str,str,str,str,cat,str,str,i64,i64,str,cat,i64,str,str,str,i64,i64,i64,str,str,str,str,str,str,str,str,str,str,str,str,i32,cat,cat
"""2361281""","""21""",13628442,"""G""","""T""","""S""","""I""","""rs905107791""","""21""","""15000763""","""21""","""13922634""","""396""","""POTED""","""ENSG00000166351""","""ENST00000299443""","""ENSP00000299443""","""Q86YR6""","""POTED_HUMAN""","""c.1187G>T""","""p.Ser396Ile""","""c.1187G>T""","""p.Ser396Ile""","""principal1""","""Y""","""1""","""YES""","""Select""","""+""","""AGC""","""2""","""0""",""".""","""./.""","""./.""","""./.""","""./.""",…,"""Sep 14, 2022""",-1,"""-""","""RCV004201236""","""MedGen:CN169374""","""not specified""","""germline""","""germline""","""GRCh38""","""NC_000021.9""","""21""",13628442,13628442,"""21q11.2""","""criteria provided, single subm…",1,"""-""","""N""","""-""",2,2361281,13628442,"""G""","""T""","""-""","""-""","""-""","""-""","""-""","""-""","""1187G>T ""","""Ser396Ile""","""NC_000021.9:13628442:G:T""","""NM_174981.6""",1,"""VUS""","""VUS"""
"""2599176""","""21""",13630941,"""G""","""T""","""V""","""F""","""rs937658823""","""21""","""15003262""","""21""","""13925133""","""415""","""POTED""","""ENSG00000166351""","""ENST00000299443""","""ENSP00000299443""","""Q86YR6""","""POTED_HUMAN""","""c.1243G>T""","""p.Val415Phe""","""c.1243G>T""","""p.Val415Phe""","""principal1""","""Y""","""1""","""YES""","""Select""","""+""","""GTT""","""1""","""0""",""".""","""./.""","""./.""","""./.""","""./.""",…,"""Jun 16, 2023""",-1,"""-""","""RCV004345528""","""MedGen:CN169374""","""not specified""","""germline""","""germline""","""GRCh38""","""NC_000021.9""","""21""",13630941,13630941,"""21q11.2""","""criteria provided, single subm…",1,"""-""","""N""","""-""",2,2599176,13630941,"""G""","""T""","""-""","""-""","""-""","""-""","""-""","""-""","""1243G>T ""","""Val415Phe""","""NC_000021.9:13630941:G:T""","""NM_174981.6""",1,"""VUS""","""VUS"""
"""2486768""","""21""",13630977,"""G""","""A""","""V""","""M""",""".""","""21""","""15003298""","""21""","""13925169""","""427""","""POTED""","""ENSG00000166351""","""ENST00000299443""","""ENSP00000299443""","""Q86YR6""","""POTED_HUMAN""","""c.1279G>A""","""p.Val427Met""","""c.1279G>A""","""p.Val427Met""","""principal1""","""Y""","""1""","""YES""","""Select""","""+""","""GTG""","""1""","""0""",""".""","""./.""","""./.""","""./.""","""./.""",…,"""Feb 17, 2023""",-1,"""-""","""RCV004275341""","""MedGen:CN169374""","""not specified""","""germline""","""germline""","""GRCh38""","""NC_000021.9""","""21""",13630977,13630977,"""21q11.2""","""criteria provided, single subm…",1,"""-""","""N""","""-""",2,2486768,13630977,"""G""","""A""","""-""","""-""","""-""","""-""","""-""","""-""","""1279G>A ""","""Val427Met""","""NC_000021.9:13630977:G:A""","""NM_174981.6""",1,"""Benign""","""Likely benign"""
"""2277

In [9]:
varchamp_clinvar_dbnsfp_df = varchamp_w_seq_clinvar.join(
    df_clinvar_dedup_struct.select(
        [c for c in df_clinvar_dedup_struct.columns if c not in varchamp_w_seq_clinvar.columns]
    ),
    left_on="spdi",
    right_on="SPDI",
    how="left"
)

In [ ]:
# varchamp_clinvar_dbnsfp_df.drop("seq", "seq_aa").write_csv("../3_outputs/VarChAMP_Allele_Collection_1Perc_ClinVar_dbNSFP.tsv", separator="\t")
# varchamp_clinvar_dbnsfp_df.write_parquet("../3_outputs/VarChAMP_Allele_Collection_1Perc_ClinVar_dbNSFP.parquet")

In [11]:
display(varchamp_clinvar_dbnsfp_df.group_by("clinvar_clnsig_clean").agg(pl.len()))
display(varchamp_clinvar_dbnsfp_df.group_by("clinvar_clnsig_clean_pp_strict").agg(pl.len()).sort(by="clinvar_clnsig_clean_pp_strict", descending=False))

clinvar_clnsig_clean,len
str,u32
"""Pathogenic""",5004
"""Benign""",717
"""Conflicting""",1146
"""Others""",1475
"""VUS""",1257


clinvar_clnsig_clean_pp_strict,len
str,u32
"""Benign""",295
"""Conflicting""",1137
"""Likely benign""",408
"""Likely pathogenic""",2060
"""Others""",1536
"""Pathogenic""",2911
"""VUS""",1252


### 2.2 Mapping pLDDT and RSA scores

In [ ]:
cd data_processing_scripts
./1_mapping_plddt_rsa_scores.py \
      --input_file "../../3_outputs/VarChAMP_Allele_Collection_1Perc_ClinVar_dbNSFP.parquet" \
      --output_file "../../3_outputs/VarChAMP_Allele_Collection_1Perc_ClinVar_dbNSFP_pLDDT_RSA.parquet" \
      --workers 8 \
      --chunk-size 1000 \
      > ../../3_outputs/processed_data/alphafold/pdb_prioritized_variants_add_plddt_rsa_scores.log 2>&1

### 2.3 Mapping protein features from G2P

In [ ]:
cd data_processing_scripts
./2_mapping_g2p_feats.py \
    --input_file "../../3_outputs/VarChAMP_Allele_Collection_1Perc_ClinVar_dbNSFP_pLDDT_RSA.parquet" \
    --output_file "../../3_outputs/VarChAMP_Allele_Collection_1Perc_ClinVar_dbNSFP_pLDDT_RSA_G2P.parquet" \
    --failure_report "../../3_outputs/g2p_mapping_failures.tsv"